# 0. les biblios

In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. Charger environmental_monthly.csv

In [ ]:
# Charger les données
df = pd.read_csv("environmental_monthly.csv")

In [ ]:
# 1. Dimensions et types de colonnes
print("Dimensions :", df.shape)
print("\nTypes de colonnes :")
df.info()

In [ ]:
# 2. Échantillon des données
print("\nPremières lignes :")
display(df.head())

- 4994 lignes et 11 colonnes.
- Une ligne = un relevé environnemental mensuel pour un site (`site_code` + `reporting_month`).
- Presque toutes les colonnes sont en `object` alors qu'elles devraient être numériques ou date : c'est déjà le signe de formats incohérents.

# 2. Doublons

### a- Doublons de lignes

In [ ]:
print("Nombre de lignes strictement dupliquées :", df.duplicated().sum())
display(df[df.duplicated(keep=False)].sort_values(['site_code', 'reporting_month']))

### b- Doublons sur la clé métier (site_code + reporting_month)

Le vrai identifiant de ce jeu de données n'est pas une colonne unique : c'est le couple **(site, mois)**.
Un même site ne peut pas avoir deux relevés pour le même mois.

In [ ]:
cle = ['site_code', 'reporting_month']
print("Couples (site_code, reporting_month) dupliqués :", df.duplicated(subset=cle).sum())
display(df[df.duplicated(subset=cle, keep=False)].sort_values(cle))

- Solution

In [ ]:
# On supprime les doublons stricts puis les doublons de clé métier (on garde la 1ère occurrence)
df = df.drop_duplicates(keep='first')
df = df.drop_duplicates(subset=cle, keep='first')

# Vérification
print("Nombre de lignes après suppression :", df.shape[0])
print("Doublons stricts restants :", df.duplicated().sum())
print("Doublons de clé restants :", df.duplicated(subset=cle).sum())

### c- Doublons de colonnes

In [ ]:
# 1. Vérifier si des noms de colonnes sont strictement identiques
noms_doublons = df.columns[df.columns.duplicated()]
print("Noms de colonnes en double :", noms_doublons.tolist())

# 2. Vérifier si le contenu de certaines colonnes est 100% identique
contenu_doublons = df.columns[df.T.duplicated()]
print("Colonnes avec un contenu dupliqué :", contenu_doublons.tolist())

- Aucune colonne redondante au sens strict.
- On vérifiera plus bas la redondance *fonctionnelle* (ex : `site_code` → `land_use_category`) et la colinéarité via la matrice de corrélation.

# 3. Formats incohérents

### Les types de chaque colonne :

In [ ]:
print(df.dtypes)

### - Vérifier le format de site_code (catégorielle)

In [ ]:
print("Valeurs uniques :", df['site_code'].nunique())
print("Formats structurels :", df['site_code'].str.replace(r'[A-Z0-9]+', 'X', regex=True).unique())
print(df['site_code'].str.extract(r'^([A-Z]+)-')[0].value_counts())
print("Manquants :", df['site_code'].isna().sum())

→ Format homogène `XXX-XXX` (64 sites répartis sur 3 régions : AVL, KNT, TMB). Rien à corriger.

### - Vérifier le format de reporting_month (date) <--->

In [ ]:
# On remplace tous les chiffres par 9 pour voir les "patrons" de format présents
print("Formats bruts rencontrés :", df['reporting_month'].str.replace(r'\d', '9', regex=True).unique())

dates = pd.to_datetime(df['reporting_month'], errors='coerce')
print("Valeurs non convertibles :", dates.isna().sum() - df['reporting_month'].isna().sum())

- Solution

In [ ]:
# Conversion en vrai type datetime (format ISO homogène ici)
df['reporting_month'] = pd.to_datetime(df['reporting_month'], format='mixed', errors='coerce')

# Vérification
print("Type de la colonne :", df['reporting_month'].dtype)
print("Dates non convertibles :", df['reporting_month'].isna().sum())
print("Période couverte :", df['reporting_month'].min(), "->", df['reporting_month'].max())
print("Nombre de mois par site :")
print(df.groupby('site_code').size().value_counts())

### - Vérifier le format des colonnes numériques <--->

In [ ]:
cols_num = ['mean_temperature_c', 'rainfall_mm', 'mean_humidity_pct',
            'vegetation_cover_pct', 'habitat_disturbance_index',
            'pesticide_index', 'data_completeness_pct']

for col in cols_num:
    s = df[col]
    non_num = s[pd.to_numeric(s, errors='coerce').isna() & s.notna()]
    print(col, "| non convertibles :", len(non_num), "| exemples :", non_num.unique()[:8])

**Constat :** trois problèmes se cumulent sur les mêmes colonnes :
- **séparateur décimal variable** : `23,81` (virgule française) au lieu de `23.81`
- **unité collée à la valeur** : `25.86 C`, `102.7 mm`
- résultat : pandas a typé toute la colonne en texte.

Comme le cours le recommande, on **garde une seule unité par colonne** (déjà indiquée dans le nom : `_c`, `_mm`, `_pct`) et on convertit tout le reste.

- Solution

In [ ]:
for col in cols_num:
    df[col] = (df[col].astype(str)
                      .str.replace(r'\s*(mm|C|%)$', '', regex=True)   # on enlève l'unité collée
                      .str.replace(',', '.', regex=False)             # virgule -> point
                      .str.strip()
                      .replace({'nan': np.nan, '': np.nan}))
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Vérification
print(df[cols_num].dtypes)
print("\nValeurs encore non numériques :")
print(df[cols_num].isna().sum())

### - Vérifier le format de land_use_category (catégorielle) <--->

In [ ]:
print(df['land_use_category'].value_counts(dropna=False))
print("Nombre de catégories brutes :", df['land_use_category'].nunique())

**Constat :** 18 catégories alors qu'il n'y en a que 8 réellement. Le même objet est écrit de 3 façons :
`karst_habitat`, `Karst_Habitat`, `KARST_HABITAT`.

Bonne pratique du cours : identifier les différents noms d'un même objet → choisir le label le plus adapté → remplacer le reste.
Ici on choisit le format `snake_case` en minuscules (le plus fréquent).

- Solution

In [ ]:
df['land_use_category'] = df['land_use_category'].str.strip().str.lower()

# Vérification
print(df['land_use_category'].value_counts())
print("Nombre de catégories après nettoyage :", df['land_use_category'].nunique())

In [ ]:
# Cohérence : un site doit toujours avoir le même type d'occupation du sol
print("Sites ayant plusieurs land_use_category :",
      (df.groupby('site_code')['land_use_category'].nunique() > 1).sum())

→ Chaque site n'a qu'une seule catégorie d'occupation du sol : la correspondance `site_code` → `land_use_category` est parfaite.
On conserve quand même les deux colonnes (identifiant du lieu vs. caractéristique écologique), comme information conceptuellement distincte.

### - Vérifier le format de weather_exception_code (catégorielle)

In [ ]:
print(df['weather_exception_code'].value_counts(dropna=False))
print("Manquants :", df['weather_exception_code'].isna().sum())

# 4. Valeurs manquantes

### a. Constat du problème

In [ ]:
print("Valeurs manquantes par colonne :")
print(df.isna().sum())

print("\nPourcentage de manquants par colonne :")
print((df.isna().sum() / len(df) * 100).round(2))

- `weather_exception_code` : ~96% de manquants
- `pesticide_index` : 15 manquants (0.3%)
- `vegetation_cover_pct` : 10 manquants (0.2%)

Les deux cas n'ont rien à voir et se traitent différemment.

### b. weather_exception_code : l'absence d'information EST une information

Hypothèse : ce n'est pas une donnée perdue, c'est un code d'exception météo qui n'est renseigné que
quand le mois est anormal. Un mois "normal" n'a donc pas de code.

In [ ]:
# Vérification de l'hypothèse : comparer la pluviométrie selon le code
print(df.groupby('weather_exception_code')['rainfall_mm'].describe())
print("\nMois sans code :")
print(df[df['weather_exception_code'].isna()]['rainfall_mm'].describe())

→ Preuve par les données : `DRY_SPELL` = mois très secs (moyenne ~24 mm), `HEAVY_RAIN` = mois très pluvieux
(moyenne ~255 mm), les lignes sans code = régime normal (~160 mm).

Le vide est donc porteur de sens. Comme le dit le cours, « considérer que l'absence d'information est une
information en soi » : on remplace `NaN` par une vraie modalité `NORMAL` plutôt que de supprimer 96% du jeu.

In [ ]:
df['weather_exception_code'] = df['weather_exception_code'].fillna('NORMAL')

# Vérification
print(df['weather_exception_code'].value_counts())
print("Manquants restants :", df['weather_exception_code'].isna().sum())

### c. vegetation_cover_pct et pesticide_index : inférence par série temporelle du site

Hypothèse : ces trous sont isolés (1 ou 2 mois par site). Comme chaque site est suivi mois par mois,
la valeur manquante peut être inférée à partir des mois qui l'encadrent **sur le même site**,
plutôt qu'avec une moyenne globale qui écraserait les différences entre sites.

In [ ]:
for col in ['vegetation_cover_pct', 'pesticide_index']:
    trous = df[df[col].isna()]
    print(col, "->", len(trous), "trous répartis sur", trous['site_code'].nunique(), "sites")
    display(trous[['site_code', 'reporting_month', col]].head())

- Solution

In [ ]:
# On trie par site et par mois, puis on interpole à l'intérieur de chaque site
df = df.sort_values(['site_code', 'reporting_month']).reset_index(drop=True)

for col in ['vegetation_cover_pct', 'pesticide_index']:
    df[col] = df.groupby('site_code')[col].transform(
        lambda s: s.interpolate(limit_direction='both')
    )

# Vérification
print("Valeurs manquantes restantes :")
print(df.isna().sum())

# 5. Gestion des variables catégorielles

### a. Création des variables temporelles

La date seule est difficile à exploiter dans un modèle : on en extrait l'année et le mois,
ce qui permettra ensuite d'observer la saisonnalité.

In [ ]:
df['year'] = df['reporting_month'].dt.year
df['month'] = df['reporting_month'].dt.month

display(df.head())

### b. Variables ordonnées

Aucune variable réellement ordinale ici : `land_use_category` (forêt, verger, karst…) et
`weather_exception_code` (DRY_SPELL / NORMAL / HEAVY_RAIN) n'ont pas de hiérarchie chiffrable
évidente — on les traite donc comme nominales.

Les "notes" (`habitat_disturbance_index`, `pesticide_index`, `data_completeness_pct`) sont déjà
des variables numériques continues, pas des catégories.

### c. Encodage des variables nominales : reporté après le merge

Ce fichier fait partie d'un lot de 6 fichiers qui seront fusionnés ensemble plus tard sur `site_code`
(et `reporting_month` pour ceux qui ont une dimension temporelle). Encoder maintenant créerait un risque :
- des colonnes dummy différentes d'un fichier à l'autre (une catégorie présente ici mais absente ailleurs) ;
- une redondance si une même colonne catégorielle (ex: `land_use_category`) existe dans plusieurs fichiers du lot.

**Décision** : on nettoie et normalise les catégories ici (fait à l'étape 3 : `land_use_category` et
`weather_exception_code` sont déjà propres, sans casse incohérente), mais on **reporte le One-Hot Encoding
au moment du merge final**, une fois tous les fichiers fusionnés, pour avoir un schéma d'encodage unique
et cohérent sur l'ensemble du projet.

In [ ]:
# Pas d'encodage ici : on garde les colonnes catégorielles nettoyées telles quelles
df_stats = df.copy()

print(df_stats.dtypes)
display(df_stats.head())

# 6. Voir les outliers (colonnes numériques)

In [ ]:
# Tableau statistique pour traquer les valeurs aberrantes
display(df_stats[cols_num].describe().T)

In [ ]:
# Configuration de la taille globale de la figure
plt.figure(figsize=(16, 10))

for i, col in enumerate(cols_num, 1):
    plt.subplot(3, 3, i)
    sns.boxplot(y=df_stats[col], color="skyblue")
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
# Comptage des points hors moustaches (règle 1.5 * IQR)
for col in cols_num:
    q1, q3 = df_stats[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    bas, haut = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df_stats[col] < bas) | (df_stats[col] > haut)).sum()
    print(f"{col:28s} min={df_stats[col].min():7.1f} max={df_stats[col].max():7.1f} -> {n} points hors moustaches")

### - Justification du traitement des outliers

- **Contrôle de plausibilité d'abord** : aucune valeur impossible n'a été trouvée (pas de négatif, pas de
code sentinelle type -99 / 999, pourcentages tous entre 0 et 100, températures entre 21 et 32 °C
cohérentes avec un climat tropical). Il n'y a donc pas d'outlier "erreur de saisie" à traiter comme
donnée manquante.

- **Décision** : les points extrêmes (mois très secs ou très pluvieux, sites très dégradés) sont
**conservés**. Ce sont des réalités environnementales, et pour plusieurs d'entre eux le jeu de données
les documente lui-même via `weather_exception_code`. Les supprimer reviendrait à effacer exactement
l'information la plus intéressante de l'étude (cf. cours : un outlier peut être une anomalie
intéressante à repérer).

# 7. Histogrammes et distributions

In [ ]:
plt.figure(figsize=(16, 12))

for i, col in enumerate(cols_num, 1):
    plt.subplot(3, 3, i)
    sns.histplot(df_stats[col], kde=True, bins=30, color='skyblue')
    plt.title(f"Distribution : {col}")

plt.tight_layout()
plt.show()

In [ ]:
# Coefficients d'asymétrie (skewness) pour objectiver la forme des distributions
print(df_stats[cols_num].skew().round(2))

### - Analyse exploratoire : distributions et lois statistiques

- **Proches d'une loi normale** : `mean_temperature_c`, `mean_humidity_pct` et `rainfall_mm` sont à peu près
symétriques autour de leur moyenne (asymétrie faible) — associer moyenne + écart-type a donc du sens.

- **Distributions asymétriques** : `vegetation_cover_pct` et `data_completeness_pct` sont étirées vers la
gauche (la plupart des sites/relevés sont bons, une minorité est très basse), tandis que
`habitat_disturbance_index` est étirée vers la droite (beaucoup de sites peu perturbés, une queue de
sites très dégradés). Pour ces variables, la **médiane + quartiles + min/max** décrit mieux la réalité
que la moyenne seule.

### - Répartition des variables catégorielles

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.countplot(y='land_use_category', data=df_stats,
              order=df_stats['land_use_category'].value_counts().index, color='skyblue')
plt.title("Répartition des types d'occupation du sol")

plt.subplot(1, 2, 2)
sns.countplot(y='weather_exception_code', data=df_stats,
              order=df_stats['weather_exception_code'].value_counts().index, color='skyblue')
plt.title("Répartition des codes d'exception météo")

plt.tight_layout()
plt.show()

### - Saisonnalité (moyenne par mois calendaire)

In [ ]:
saison = df_stats.groupby('month')[['mean_temperature_c', 'rainfall_mm', 'mean_humidity_pct']].mean()
display(saison.round(1))

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(saison.index, saison['mean_temperature_c'], marker='o', color='tomato', label='Température (°C)')
ax1.set_xlabel("Mois")
ax1.set_ylabel("Température moyenne (°C)", color='tomato')

ax2 = ax1.twinx()
ax2.bar(saison.index, saison['rainfall_mm'], alpha=0.3, color='steelblue', label='Pluie (mm)')
ax2.set_ylabel("Pluviométrie moyenne (mm)", color='steelblue')

plt.title("Saisonnalité : température vs pluviométrie")
plt.tight_layout()
plt.show()

→ Signal clair de mousson : les mois les plus pluvieux (août–octobre) sont aussi les plus frais,
les mois secs (février–avril) les plus chauds. C'est cohérent avec un climat tropical et confirme que
les données ne sont pas aléatoires.

# 8. Corrélations

### Matrices de corrélation (Pearson + Spearman) sur les variables numériques

In [ ]:
# 1. On calcule explicitement les DEUX méthodes distinctes
matrice_spearman = df_stats[cols_num].corr(method='spearman')
matrice_pearson = df_stats[cols_num].corr(method='pearson')

display(matrice_spearman.round(2))

In [ ]:
# Affichage de Spearman
plt.figure(figsize=(9, 7))
sns.heatmap(matrice_spearman, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)
plt.title("Matrice de corrélation de Spearman")
plt.tight_layout()
plt.show()

# Affichage de Pearson
plt.figure(figsize=(9, 7))
sns.heatmap(matrice_pearson, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)
plt.title("Matrice de corrélation de Pearson")
plt.tight_layout()
plt.show()

### Nuages de points croisés (Pairplot)

In [ ]:
variables_a_croiser = ['rainfall_mm', 'mean_humidity_pct', 'mean_temperature_c', 'vegetation_cover_pct']

sns.pairplot(df_stats[variables_a_croiser], kind='scatter',
             plot_kws={'alpha': 0.3, 'color': 'teal', 's': 15})
plt.suptitle("Nuages de points croisés des variables climatiques", y=1.02)
plt.show()

### - Analyse des corrélations

- **pluie ↔ humidité (≈ 0.75)** : la corrélation la plus forte, relation monotone et quasi linéaire.
Les deux variables mesurent en partie le même phénomène → redondance à garder en tête si on construit
un modèle (colinéarité).
- **température ↔ pluie / humidité (≈ -0.22 / -0.27)** : négatif et modéré, c'est la saisonnalité vue plus haut.
- **couverture végétale ↔ perturbation de l'habitat (≈ -0.31)** : plus un site est perturbé, moins il est végétalisé.
Relation attendue écologiquement, mais **corrélation n'est pas causalité** : les deux peuvent découler
du même facteur (type d'occupation du sol).
- **`pesticide_index` et `data_completeness_pct`** ne sont corrélés à rien : `data_completeness_pct` est une
métadonnée de qualité de collecte, pas une mesure environnementale.
- **Pearson vs Spearman** donnent ici des valeurs très proches : les relations sont plutôt monotones et
linéaires, et il n'y a pas de queue de distribution extrême qui tire les coefficients (contrairement à des
données de comptage).

# 9. Export du jeu de données nettoyé

On exporte une seule version : nettoyée, mais **non encodée**. L'encodage One-Hot des catégorielles
sera fait plus tard, une seule fois, sur le jeu de données fusionné avec les 5 autres fichiers du lot.

In [ ]:
df_stats.to_csv("environmental_monthly_clean.csv", index=False)

print("Dimensions finales :", df_stats.shape)
print("Valeurs manquantes restantes :", df_stats.isna().sum().sum())

# 10. Synthèse du nettoyage

| Problème identifié | Colonnes concernées | Traitement appliqué |
|---|---|---|
| Doublons stricts (2) et doublons de clé (site, mois) | toutes | suppression, on garde la 1ère occurrence |
| Date en texte | `reporting_month` | conversion en `datetime` + extraction `year` / `month` |
| Séparateur décimal virgule | températures, pluie, humidité | remplacement `,` → `.` |
| Unité collée à la valeur (`25.8 C`, `102.7 mm`) | `mean_temperature_c`, `rainfall_mm` | suppression du suffixe, une seule unité par colonne |
| Casse incohérente (18 catégories pour 8 réelles) | `land_use_category` | normalisation en minuscules |
| Manquants "porteurs de sens" (96%) | `weather_exception_code` | remplacés par la modalité `NORMAL` |
| Trous isolés dans les séries | `vegetation_cover_pct`, `pesticide_index` | interpolation temporelle **par site** |
| Valeurs extrêmes | variables numériques | **conservées** (réalités environnementales, pas des erreurs) |
| Encodage des catégorielles | `land_use_category`, `weather_exception_code` | **reporté** : fait une seule fois après le merge des 6 fichiers |

**Biais introduits et assumés** (cf. cours : nettoyer = choisir = introduire des biais) :
- l'interpolation invente 25 valeurs qui n'ont jamais été mesurées (0.3% du jeu, impact négligeable mais réel) ;
- en gardant la 1ère occurrence des doublons, on suppose que les deux lignes étaient identiques — ce qui a
été vérifié ici ;
- transformer l'absence de code météo en `NORMAL` suppose que le protocole de saisie était fiable : si un
opérateur a simplement oublié de remplir le champ un mois anormal, ce mois est désormais classé "normal".